# playground_checkpoints_vitb32 — when do the top head-pairs emerge?

Track the final top-10 (vision-head, text-head) pairs across 5 checkpoints of the single B/32 laion-2B run `laion/scaling-laws-openclip .../Model-B-32_Data-2B_Samples-13B_lr-5e-4_bs-32k`, at samples ~ {0(init), n/4, n/2, 3n/4, last}.

Per checkpoint we run ONLY the two proven extraction scripts (via `--pretrained <local .pt>`, isolated `--output_dir`, same `--seed`), then derive accuracy / C_hat / PC-subspace overlap in-notebook. **No script edits.**

Question: do aligned pairs appear **suddenly** or **continuously**, are they there at **init**, and are the top heads **consistent**.

## Stage 0 — setup (imports/params/model, from playground_all) + pick & download 5 checkpoints

In [1]:
## Imports
import numpy as np
import torch
import json
import os
from tabulate import tabulate
from PIL import Image
from torch.nn import functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision.datasets import ImageNet, CIFAR10, CIFAR100, ImageFolder
from utils.misc.misc import accuracy, accuracy_correct
from utils.scripts.algorithms_text_explanations import svd_data_approx
from utils.scripts.algorithms_text_explanations_funcs import *
from utils.models.factory import create_model_and_transforms, get_tokenizer
from utils.models.prs_hook import hook_prs_logger
from utils.models.prs_hook_text import hook_prs_logger_text
from utils.misc.visualization import visualization_preprocess
from utils.datasets_constants.imagenet_classes import imagenet_classes
from utils.datasets_constants.cifar_10_classes import cifar_10_classes
from utils.datasets_constants.cub_classes import cub_classes, waterbird_classes
from utils.datasets.dataset_helpers import dataset_subset
from utils.datasets.binary_waterbirds import BinaryWaterbirds


/home/virgilio/lib/minconda3/envs/MT/lib/python3.10/site-packages/torch/cuda/__init__.py:51: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
## Parameters
device = 'cuda'
model_name = 'ViT-B-32'        # 'ViT-H-14', 'ViT-L-14', 'ViT-B-16', 'ViT-B-32'
seed = 0
num_last_layers_ = 11            # how many of the LAST layers get a full PC decomposition, for BOTH encoders
algorithm = "svd_data_approx"

# Dataset the vision is run on (both for the activations / visualization)
dataset_image_name = "imagenet"      
subset_dim = 10 
tot_samples_per_class = 10 # nr samples per class
path = './datasets/'

# Dataset the text encoder is run to produce activations
dataset = "imagenet_descriptions_personal" 
native_per_class = 10           # sentences per class in "dataset"
sentences_per_class = 1         # keep the LAST k per class; 1 -> only the class-name sentence

# Shared: labels used to text-label PCs of BOTH encoders
dataset_text_name = "top_1500_nouns_5_sentences_imagenet_clean"

cache_dir = "../cache"

if model_name == "ViT-H-14":
    pretrained = "laion2B-s32B-b79K"; precision = "fp32"
elif model_name == "ViT-L-14":
    pretrained = "laion2B-s32B-b82K"; precision = "fp32"
elif model_name == "ViT-B-16":
    pretrained = "laion2B-s34B-b88K"; precision = "fp32"
elif model_name == "ViT-B-32":
    pretrained = "laion2B-s34B-b79K"; precision = "fp32"
elif model_name == "ViT-L-14-336":
    pretrained = "openai"; precision = "fp16"


In [3]:
## Loading Model — one CLIP checkpoint, both towers
model, _, preprocess = create_model_and_transforms(
    model_name, pretrained=pretrained, precision=precision, cache_dir=cache_dir)
model.to(device)
model.eval()
tokenizer = get_tokenizer(model_name)

print("Model parameters:", f"{np.sum([int(np.prod(p.shape)) for p in model.parameters()]):,}")
print("Number of VISION-encoder layers:", len(model.visual.transformer.resblocks))
print("Number of TEXT-encoder layers:", len(model.transformer.resblocks))


Using local files
Model parameters: 151,277,313
Number of VISION-encoder layers: 12
Number of TEXT-encoder layers: 12


In [4]:
# --- pick checkpoints: first-after-init + 2 early (pre-quarter) + n/4,n/2,3n/4,last ---
from huggingface_hub import list_repo_files, hf_hub_download
REPO = "laion/scaling-laws-openclip"
RUN  = "full_checkpoints/Model-B-32_Data-2B_Samples-13B_lr-5e-4_bs-32k"
ck = sorted(f for f in list_repo_files(REPO) if f.startswith(RUN) and f.endswith(".pt"))
print(f"{len(ck)} checkpoints in the run:"); [print("  ", f) for f in ck]
# VERIFY the printed order == increasing samples-seen (filenames usually carry the sample/step count)
import numpy as np
# dense early (emergence is usually fast): 0=first-after-init, 0.08 & 0.16 = pre-quarter
frac = [0.0, 0.08, 0.16, 0.25, 0.5, 0.75, 1.0]
idx  = sorted(set(int(round(f*(len(ck)-1))) for f in frac))   # dedupe if cadence is coarse
sel  = [ck[i] for i in idx]
sel_paths = [hf_hub_download(REPO, f, cache_dir=cache_dir) for f in sel]
print("\nselected trajectory checkpoints:"); [print("  ", f) for f in sel]


66 checkpoints in the run:
   full_checkpoints/Model-B-32_Data-2B_Samples-13B_lr-5e-4_bs-32k/epoch_1.pt
   full_checkpoints/Model-B-32_Data-2B_Samples-13B_lr-5e-4_bs-32k/epoch_10.pt
   full_checkpoints/Model-B-32_Data-2B_Samples-13B_lr-5e-4_bs-32k/epoch_11.pt
   full_checkpoints/Model-B-32_Data-2B_Samples-13B_lr-5e-4_bs-32k/epoch_12.pt
   full_checkpoints/Model-B-32_Data-2B_Samples-13B_lr-5e-4_bs-32k/epoch_13.pt
   full_checkpoints/Model-B-32_Data-2B_Samples-13B_lr-5e-4_bs-32k/epoch_14.pt
   full_checkpoints/Model-B-32_Data-2B_Samples-13B_lr-5e-4_bs-32k/epoch_15.pt
   full_checkpoints/Model-B-32_Data-2B_Samples-13B_lr-5e-4_bs-32k/epoch_16.pt
   full_checkpoints/Model-B-32_Data-2B_Samples-13B_lr-5e-4_bs-32k/epoch_17.pt
   full_checkpoints/Model-B-32_Data-2B_Samples-13B_lr-5e-4_bs-32k/epoch_18.pt
   full_checkpoints/Model-B-32_Data-2B_Samples-13B_lr-5e-4_bs-32k/epoch_19.pt
   full_checkpoints/Model-B-32_Data-2B_Samples-13B_lr-5e-4_bs-32k/epoch_2.pt
   full_checkpoints/Model-B-32_Data-2B_

[None, None, None, None]

In [5]:
# --- init (samples=0): use a repo init ckpt if present, else fresh open_clip B/32 (default init = run start) ---
init_cand = [f for f in ck if any(k in f.lower() for k in ("step_0","epoch_0","_init","samples-0_","_s0_"))]
if init_cand:
    init_path = hf_hub_download(REPO, init_cand[0], cache_dir=cache_dir); print("repo init ckpt:", init_cand[0])
else:
    m0, _, _ = create_model_and_transforms(model_name, pretrained=None, precision=precision, cache_dir=cache_dir)
    init_path = f"{cache_dir}/{model_name}_init_seed{seed}.pt"; torch.save(m0.state_dict(), init_path); del m0
    print("no repo init -> saved fresh default-init B/32 ->", init_path)

CKPTS = [("s0_init", init_path)] + [(f"s{i+1}", p) for i, p in enumerate(sel_paths)]   # 5 total
for tag, p in CKPTS: print(f"  {tag:8} {p}")


no repo init -> saved fresh default-init B/32 -> ../cache/ViT-B-32_init_seed0.pt
  s0_init  ../cache/ViT-B-32_init_seed0.pt
  s1       ../cache/models--laion--scaling-laws-openclip/snapshots/ff857939164def1f6a27e2403ad5b978e1a8f839/full_checkpoints/Model-B-32_Data-2B_Samples-13B_lr-5e-4_bs-32k/epoch_24.pt
  s2       ../cache/models--laion--scaling-laws-openclip/snapshots/ff857939164def1f6a27e2403ad5b978e1a8f839/full_checkpoints/Model-B-32_Data-2B_Samples-13B_lr-5e-4_bs-32k/epoch_39.pt
  s3       ../cache/models--laion--scaling-laws-openclip/snapshots/ff857939164def1f6a27e2403ad5b978e1a8f839/full_checkpoints/Model-B-32_Data-2B_Samples-13B_lr-5e-4_bs-32k/epoch_54.pt
  s4       ../cache/models--laion--scaling-laws-openclip/snapshots/ff857939164def1f6a27e2403ad5b978e1a8f839/full_checkpoints/Model-B-32_Data-2B_Samples-13B_lr-5e-4_bs-32k/epoch_latest.pt


## Stage 1 — per-checkpoint extraction (the only heavy step)
Runs vision + text extraction into `output_dir/ck_<tag>/` with the checkpoint weights, same seed (same eval subset). Skips a checkpoint whose outputs already exist.

**Risk to verify on first run:** the scaling-laws `.pt` are training-state dicts; if `load_checkpoint(strict=True)` rejects extra keys (e.g. `logit_scale`, `module.` prefix), the extraction errors — then we pre-strip the state_dict.

In [6]:
import os, subprocess
RE_EXTRACT = False   # set True to force re-extraction even if cached activations exist

def sh(cmd):
    print("$", cmd); r = subprocess.run(cmd, shell=True); assert r.returncode == 0, f"failed: {cmd}"

def have(od, prefix, suffix):   # all outputs Stage 2 will load must exist (attn+mlp+labels)
    return all(os.path.exists(f"{od}/{prefix}_{k}{suffix}_{model_name}_seed_{seed}.npy")
               for k in ("attn", "mlp", "labels"))

def extract(tag, ckpt):
    od = f"output_dir/ck_{tag}"; os.makedirs(od, exist_ok=True)
    common = f"--model {model_name} --pretrained {ckpt} --seed {seed} --device {device} --output_dir {od} --cache_dir {cache_dir}"
    # vision tower  (compute_activation_values takes --dataset)
    if not RE_EXTRACT and have(od, dataset_image_name, ""):
        print(f"[{tag}] vision cached, skipping")
    else:
        sh(f"python -m utils.scripts.compute_activation_values --dataset {dataset_image_name} "
           f"--batch_size 64 --data_path {path} --samples_per_class {subset_dim} "
           f"--tot_samples_per_class {tot_samples_per_class} {common}")
    # text tower  (compute_activation_values_text takes the corpus via --text_descriptions, no --dataset)
    if not RE_EXTRACT and have(od, dataset, "_text"):
        print(f"[{tag}] text cached, skipping")
    else:
        sh(f"python -m utils.scripts.compute_activation_values_text --text_descriptions {dataset} "
           f"--batch_size 256 --native_per_class {native_per_class} "
           f"--sentences_per_class {sentences_per_class} {common}")

for tag, ck in CKPTS: extract(tag, ck)
print("extraction done for all checkpoints")


[s0_init] vision cached
$ python -m utils.scripts.compute_activation_values_text --text_descriptions imagenet_descriptions_personal --batch_size 256 --native_per_class 10 --sentences_per_class 1 --model ViT-B-32 --pretrained ../cache/ViT-B-32_init_seed0.pt --seed 0 --device cuda --output_dir output_dir/ck_s0_init --cache_dir ../cache


/home/virgilio/lib/minconda3/envs/MT/lib/python3.10/site-packages/torch/cuda/__init__.py:51: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Model parameters: 151,277,313
Context length: 77
Number of text layers: 12
Decomposing 1000 texts from 'imagenet_descriptions_personal' (1000 classes x 1 of 10 sentences/class).


100%|██████████| 4/4 [00:00<00:00,  5.71it/s]



Concatenating chunk files into final .npy arrays...
Final single-file arrays created:
  output_dir/ck_s0_init/imagenet_descriptions_personal_attn_text_ViT-B-32_seed_0.npy  shape (1000, 12, 8, 512)
  output_dir/ck_s0_init/imagenet_descriptions_personal_mlp_text_ViT-B-32_seed_0.npy  shape (1000, 13, 512)
  output_dir/ck_s0_init/imagenet_descriptions_personal_labels_text_ViT-B-32_seed_0.npy  shape (1000,)
Chunk files removed. Done.
$ python -m utils.scripts.compute_activation_values --dataset imagenet --batch_size 64 --data_path ./datasets/ --samples_per_class 10 --tot_samples_per_class 10 --model ViT-B-32 --pretrained ../cache/models--laion--scaling-laws-openclip/snapshots/ff857939164def1f6a27e2403ad5b978e1a8f839/full_checkpoints/Model-B-32_Data-2B_Samples-13B_lr-5e-4_bs-32k/epoch_24.pt --seed 0 --device cuda --output_dir output_dir/ck_s1 --cache_dir ../cache


...


Concatenating chunk files into final .npy arrays...
['output_dir/ck_s4/imagenet_attn_ViT-B-32_seed_0_chunk0.npy', 'output_dir/ck_s4/imagenet_attn_ViT-B-32_seed_0_chunk1.npy', 'output_dir/ck_s4/imagenet_attn_ViT-B-32_seed_0_chunk2.npy', 'output_dir/ck_s4/imagenet_attn_ViT-B-32_seed_0_chunk3.npy', 'output_dir/ck_s4/imagenet_attn_ViT-B-32_seed_0_chunk4.npy', 'output_dir/ck_s4/imagenet_attn_ViT-B-32_seed_0_chunk5.npy', 'output_dir/ck_s4/imagenet_attn_ViT-B-32_seed_0_chunk6.npy', 'output_dir/ck_s4/imagenet_attn_ViT-B-32_seed_0_chunk7.npy', 'output_dir/ck_s4/imagenet_attn_ViT-B-32_seed_0_chunk8.npy', 'output_dir/ck_s4/imagenet_attn_ViT-B-32_seed_0_chunk9.npy', 'output_dir/ck_s4/imagenet_attn_ViT-B-32_seed_0_chunk10.npy', 'output_dir/ck_s4/imagenet_attn_ViT-B-32_seed_0_chunk11.npy', 'output_dir/ck_s4/imagenet_attn_ViT-B-32_seed_0_chunk12.npy', 'output_dir/ck_s4/imagenet_attn_ViT-B-32_seed_0_chunk13.npy', 'output_dir/ck_s4/imagenet_attn_ViT-B-32_seed_0_chunk14.npy', 'output_dir/ck_s4/imagenet

/home/virgilio/lib/minconda3/envs/MT/lib/python3.10/site-packages/torch/cuda/__init__.py:51: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Model parameters: 151,277,313
Context length: 77
Number of text layers: 12
Decomposing 1000 texts from 'imagenet_descriptions_personal' (1000 classes x 1 of 10 sentences/class).


100%|██████████| 4/4 [00:01<00:00,  3.39it/s]



Concatenating chunk files into final .npy arrays...
Final single-file arrays created:
  output_dir/ck_s4/imagenet_descriptions_personal_attn_text_ViT-B-32_seed_0.npy  shape (1000, 12, 8, 512)
  output_dir/ck_s4/imagenet_descriptions_personal_mlp_text_ViT-B-32_seed_0.npy  shape (1000, 13, 512)
  output_dir/ck_s4/imagenet_descriptions_personal_labels_text_ViT-B-32_seed_0.npy  shape (1000,)
Chunk files removed. Done.
extraction done for all checkpoints


## Stage 2 — load per-checkpoint activations (both towers)

In [7]:
def load_ck(tag):
    od = f"output_dir/ck_{tag}"
    L = lambda p: torch.tensor(np.load(p, mmap_mode="r")).float()
    return dict(
        av=L(f"{od}/{dataset_image_name}_attn_{model_name}_seed_{seed}.npy"),
        mv=L(f"{od}/{dataset_image_name}_mlp_{model_name}_seed_{seed}.npy"),
        lv=torch.tensor(np.load(f"{od}/{dataset_image_name}_labels_{model_name}_seed_{seed}.npy")).long(),
        at=L(f"{od}/{dataset}_attn_text_{model_name}_seed_{seed}.npy"),
        mt=L(f"{od}/{dataset}_mlp_text_{model_name}_seed_{seed}.npy"),
        lt=torch.tensor(np.load(f"{od}/{dataset}_labels_text_{model_name}_seed_{seed}.npy")).long())

CK = {tag: load_ck(tag) for tag, _ in CKPTS}
for tag in CK: print(tag, "vision", tuple(CK[tag]["av"].shape), "text", tuple(CK[tag]["at"].shape))


s0_init vision (10000, 12, 12, 512) text (1000, 12, 8, 512)
s1 vision (10000, 12, 12, 512) text (1000, 12, 8, 512)
s2 vision (10000, 12, 12, 512) text (1000, 12, 8, 512)
s3 vision (10000, 12, 12, 512) text (1000, 12, 8, 512)
s4 vision (10000, 12, 12, 512) text (1000, 12, 8, 512)


## Stage 3 — helpers + reference top-10 pairs (fixed at the LAST checkpoint)

In [8]:
# ---- metric B (PVE-weighted PC-subspace overlap x mean norms) + accuracy, all in-notebook ----
pair_k, cutoff = 50, 0.99

def head_bundles(attns):
    # (l,h) -> (V [P,d] unit PCs, pve [P], mean_norm)  from centered per-head contributions.
    # PCs via eigh of the [d,d] Gram matrix (robust to the svd non-convergence on near-constant
    # heads, e.g. at init; eigenvalues = squared singular values).
    N, L, H, d = attns.shape
    B = {}
    for l in range(L):
        for h in range(H):
            X = attns[:, l, h]
            mn = X.norm(dim=-1).mean().item()
            Xc = X - X.mean(0)
            evals, evecs = torch.linalg.eigh(Xc.T @ Xc)          # ascending [d], columns [d,d]
            evals = evals.clamp(min=0).flip(0)                   # -> descending
            V = evecs.flip(1).T                                  # rows = PCs, descending
            tot = evals.sum()
            if tot < 1e-12:                                      # constant head (e.g. at init)
                B[(l, h)] = (V[:1], torch.zeros(1), mn); continue
            pve = evals / tot
            P = min(pair_k, int((pve.cumsum(0) < cutoff).sum().item()) + 1)
            B[(l, h)] = (V[:P], pve[:P], mn)
    return B

_bcache = {}
def bundles(tag, which):                                    # cache SVDs per (checkpoint, tower)
    key = (tag, which)
    if key not in _bcache: _bcache[key] = head_bundles(CK[tag][which])
    return _bcache[key]

def score_B(bv, bt):
    kv, kt = sorted(bv), sorted(bt)
    SA = torch.zeros(len(kv), len(kt))
    for i, ki in enumerate(kv):
        Vi, ai, _ = bv[ki]
        for j, kj in enumerate(kt):
            Vj, aj, _ = bt[kj]
            SA[i, j] = ai @ (Vi @ Vj.T).abs() @ aj          # metric A = sum PVE_i PVE_j |cos|
    nv = torch.tensor([bv[k][2] for k in kv]); nt = torch.tensor([bt[k][2] for k in kt])
    return SA * nv[:, None] * nt[None, :], kv, kt           # metric B = A * ||c_i|| ||c_j||

def subspace_overlap(A, B, k=8):                            # unweighted geometry witness
    m = min(k, A.shape[0], B.shape[0])
    return (torch.linalg.svdvals(A[:m] @ B[:m].T) ** 2).mean().item()

def zeroshot(d):
    Mv = d["av"].sum((1,2)) + d["mv"].sum(1); Mt = d["at"].sum((1,2)) + d["mt"].sum(1)
    Cn = int(max(d["lv"].max(), d["lt"].max())) + 1
    proto = torch.stack([Mt[d["lt"] == c].mean(0) for c in range(Cn)])
    proto = proto / proto.norm(dim=-1, keepdim=True); Mv = Mv / Mv.norm(dim=-1, keepdim=True)
    return (Mv @ proto.T).argmax(1).eq(d["lv"]).float().mean().item() * 100

# reference top-10 pairs by metric B at the LAST checkpoint
last = CKPTS[-1][0]
SB_last, kv, kt = score_B(bundles(last, "av"), bundles(last, "at"))
flat = SB_last.flatten().topk(10).indices
top_pairs = [(kv[int(i // len(kt))], kt[int(i % len(kt))]) for i in flat.tolist()]   # ((lv,hv),(lt,ht))
print("top-10 reference pairs by metric B (vision L,H <-> text L,H):")
for (pv, pt) in top_pairs:
    print(f"  V{pv} <-> T{pt}   B={SB_last[kv.index(pv), kt.index(pt)]:.4f}")


NameError: name 'zeroshot' is not defined

## Stage 4 — track the 10 pairs across checkpoints + accuracy

In [ ]:
samples_label = [t for t, _ in CKPTS]
acc = [zeroshot(CK[t]) for t, _ in CKPTS]
print("zero-shot accuracy per checkpoint:", [round(a, 1) for a in acc])

metricB = np.zeros((len(CKPTS), len(top_pairs)))
overlap = np.zeros((len(CKPTS), len(top_pairs)))
for c, (t, _) in enumerate(CKPTS):
    SB, kvc, ktc = score_B(bundles(t, "av"), bundles(t, "at"))
    for p, (pv, pt) in enumerate(top_pairs):
        metricB[c, p] = SB[kvc.index(pv), ktc.index(pt)].item()
        overlap[c, p] = subspace_overlap(bundles(t, "av")[pv][0], bundles(t, "at")[pt][0])

fig, axes = plt.subplots(1, 2, figsize=(15, 5)); x = np.arange(len(CKPTS))
for p in range(len(top_pairs)):
    axes[0].plot(x, metricB[:, p], marker='o', alpha=0.7,
                 label=f"V{top_pairs[p][0]}<->T{top_pairs[p][1]}")
    axes[1].plot(x, overlap[:, p], marker='o', alpha=0.7)
for ax, ttl, ylb in [(axes[0], "metric B (norm x PVE-weighted overlap)", "score_B"),
                     (axes[1], "PC-subspace overlap (unweighted, geometry only)", "mean cos^2")]:
    axt = ax.twinx(); axt.plot(x, acc, 'k--', lw=2); axt.set_ylabel("accuracy (%)")
    ax.set_xticks(x); ax.set_xticklabels(samples_label, rotation=30)
    ax.set_title(f"{ttl} of top-10 pairs vs training"); ax.set_ylabel(ylb); ax.grid(True, alpha=0.3)
axes[0].legend(fontsize=6, ncol=2)
fig.suptitle("Emergence of the top metric-B head-pairs (init = baseline; dashed = zero-shot acc)")
plt.tight_layout(); plt.show()


## Stage 5 — sudden vs continuous + consistency of the top heads

In [ ]:
# sudden vs continuous on metric B (largest single-step jump / total rise)
rise = metricB[-1] - metricB[0]
jump = np.max(np.diff(metricB, axis=0), axis=0)
print(f"{'pair':<28}{'init':>8}{'last':>8}{'rise':>8}{'maxjump/rise':>14}")
for p, (pv, pt) in enumerate(top_pairs):
    frac = jump[p] / rise[p] if rise[p] > 1e-9 else float('nan')
    print(f"  V{pv!s:<8}<->T{pt!s:<9}{metricB[0,p]:>8.3f}{metricB[-1,p]:>8.3f}{rise[p]:>8.3f}"
          f"{frac:>10.2f}  {'SUDDEN' if frac > 0.6 else 'continuous'}")

# consistency: recompute top-10 by B at each ckpt, overlap with the final set
final_set = set(top_pairs)
print("\ntop-10 (metric B) overlap with final set:")
for t, _ in CKPTS:
    SB, kvc, ktc = score_B(bundles(t, "av"), bundles(t, "at"))
    idx = SB.flatten().topk(10).indices
    s = {(kvc[int(i // len(ktc))], ktc[int(i % len(ktc))]) for i in idx.tolist()}
    print(f"  {t:8}: |cap final|={len(s & final_set)}/10")


## Stage 6 — per-checkpoint diagnostics (from playground_exploration)
For every checkpoint: architecture map of the top-10 pairs, metric-1(geometry)-vs-metric-2(influence) scatter + residual-norm weights + both-metrics bars, the E6b pair-ablation curve, and the top-pair metric-B score + cumulative share. `align_geo`=metric A, `align_infl`=metric B / (res-norms) (same ranking).

In [ ]:
import matplotlib.patches as mpatches

def pair_matrices(tag):
    bv, bt = bundles(tag, "av"), bundles(tag, "at")
    kv, kt = sorted(bv), sorted(bt)
    A = np.zeros((len(kv), len(kt)))
    for i, ki in enumerate(kv):
        Vi, ai, _ = bv[ki]
        for j, kj in enumerate(kt):
            Vj, aj, _ = bt[kj]
            A[i, j] = (ai[:, None] * aj[None, :] * (Vi @ Vj.T).abs()).sum().item()   # metric A (geometry)
    nv = np.array([bv[k][2] for k in kv]); nt = np.array([bt[k][2] for k in kt])
    d = CK[tag]
    rv = (d["av"].sum((1,2)) + d["mv"].sum(1)).norm(dim=-1).mean().item()
    rt = (d["at"].sum((1,2)) + d["mt"].sum(1)).norm(dim=-1).mean().item()
    infl = A * (nv[:, None] / rv) * (nt[None, :] / rt)                                # metric B / res-norms
    return A, infl, kv, kt, nv/rv, nt/rt

def greedy_unique(infl, kv, kt, n=10):
    order = np.argsort(-infl.ravel())
    uv, ut, out = set(), set(), []
    for f in order:
        i, j = f // len(kt), f % len(kt); v, t = kv[i], kt[j]
        if v in uv or t in ut: continue
        uv.add(v); ut.add(t); out.append((v, t))
        if len(out) == n: break
    return out

def draw_tower(ax, nrL, nrH, late, ranks, title):
    for l in range(nrL):
        for h in range(nrH):
            c = "lightgrey" if l not in late else ("#d62728" if (l, h) in ranks else "#2ca02c")
            ax.add_patch(mpatches.Rectangle((h, l), 0.9, 0.9, color=c))
            if (l, h) in ranks:
                ax.text(h+0.45, l+0.45, str(ranks[(l, h)]), ha="center", va="center", color="white", fontsize=8, fontweight="bold")
    ax.set_xlim(-0.3, nrH+0.3); ax.set_ylim(-0.3, nrL); ax.set_title(title); ax.set_xlabel("head"); ax.set_ylabel("layer")

def mean_ablate(attns, base, heads):
    e = base.clone()
    for (l, h) in heads: e = e + attns[:, l, h].mean(0) - attns[:, l, h]
    return e

def ablation_curves(tag, top_infl, kv, kt, infl):
    d = CK[tag]
    Mv = d["av"].sum((1,2)) + d["mv"].sum(1); Mt = d["at"].sum((1,2)) + d["mt"].sum(1)
    Cn = int(max(d["lv"].max(), d["lt"].max())) + 1
    proto = lambda M: (lambda P: P / P.norm(dim=-1, keepdim=True))(torch.stack([M[d["lt"] == c].mean(0) for c in range(Cn)]))
    protoT0 = proto(Mt); imgs0 = Mv / Mv.norm(dim=-1, keepdim=True)
    def acc(order, k):
        vh = [p[0] for p in order[:k]]; th = [p[1] for p in order[:k]]
        ev = mean_ablate(d["av"], Mv, vh); ev = ev / ev.norm(dim=-1, keepdim=True)
        aV = (ev @ protoT0.T).argmax(1).eq(d["lv"]).float().mean().item() * 100
        aT = (imgs0 @ proto(mean_ablate(d["at"], Mt, th)).T).argmax(1).eq(d["lv"]).float().mean().item() * 100
        return (aV + aT) / 2
    allp = [(v, t) for v in kv for t in kt]
    rank = [(kv[f // len(kt)], kt[f % len(kt)]) for f in np.argsort(-infl.ravel())]
    lay = sorted(allp, key=lambda p: (p[0][0], p[0][1], p[1][0], p[1][1]))
    rng = np.random.default_rng(1); rnd = [allp[i] for i in rng.choice(len(allp), min(40, len(allp)), replace=False)]
    ms = [0, 2, 5, 10, 20]
    return ms, {"top": [acc(rank, k) for k in ms], "bottom": [acc(rank[::-1], k) for k in ms],
                "random": [acc(rnd, k) for k in ms], "layer_order": [acc(lay, k) for k in ms]}

def report(tag):
    A, infl, kv, kt, wv_all, wt_all = pair_matrices(tag)
    top = greedy_unique(infl, kv, kt, 10)
    print(f"[{tag}] top-10 pairs (metric B, may be noise at init):",
          [(f"V{v}", f"T{t}") for v, t in top])
    nrLv, nrHv = CK[tag]["av"].shape[1], CK[tag]["av"].shape[2]
    nrLt, nrHt = CK[tag]["at"].shape[1], CK[tag]["at"].shape[2]
    lateV = set(range(nrLv - num_last_layers_, nrLv)); lateT = set(range(nrLt - num_last_layers_, nrLt))
    rv = {v: r for r, (v, t) in enumerate(top, 1)}; rt = {t: r for r, (v, t) in enumerate(top, 1)}

    fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
    draw_tower(ax[0], nrLv, nrHv, lateV, rv, "Vision tower"); draw_tower(ax[1], nrLt, nrHt, lateT, rt, "Text tower")
    fig.legend(handles=[mpatches.Patch(color="#d62728", label="paired head (rank)"),
                        mpatches.Patch(color="#2ca02c", label="decomposed, unpaired"),
                        mpatches.Patch(color="lightgrey", label="mean-ablated upstream")], loc="upper center", ncol=3)
    fig.suptitle(f"[{tag}] top-10 metric-B pairs"); plt.tight_layout(rect=[0,0,1,0.92]); plt.show()

    la_top = np.array([A[kv.index(v), kt.index(t)] for v, t in top])
    law_top = np.array([infl[kv.index(v), kt.index(t)] for v, t in top])
    wv = np.array([wv_all[kv.index(v)] for v, t in top]); wt = np.array([wt_all[kt.index(t)] for v, t in top])
    r = np.arange(1, len(top) + 1)
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
    ax[0].scatter(A.ravel(), infl.ravel(), s=6, c="lightgrey"); ax[0].scatter(la_top, law_top, s=40, c="#d62728")
    for k, (x, y) in enumerate(zip(la_top, law_top), 1): ax[0].annotate(str(k), (x, y), fontsize=8)
    ax[0].set_xlabel("metric A (geometry)"); ax[0].set_ylabel("metric B (influence)"); ax[0].set_title(f"[{tag}] A vs B")
    ax[1].bar(r-0.2, wv, 0.4, label="w_v"); ax[1].bar(r+0.2, wt, 0.4, label="w_t")
    ax[1].set_xlabel("pair rank"); ax[1].set_ylabel("|h|/|res|"); ax[1].set_title("residual-norm weights"); ax[1].legend()
    ax[2].bar(r-0.2, la_top/la_top.max(), 0.4, label="A (norm.)"); ax[2].bar(r+0.2, law_top/law_top.max(), 0.4, label="B (norm.)")
    ax[2].set_xlabel("pair rank"); ax[2].set_title("both metrics per pair"); ax[2].legend()
    plt.tight_layout(); plt.show()

    ms, cur = ablation_curves(tag, top, kv, kt, infl)
    plt.figure(figsize=(6, 4))
    for lab, c in cur.items(): plt.plot(ms, c, "o-", label=lab)
    plt.legend(); plt.xlabel("# ablated head-pairs"); plt.ylabel("mean zero-shot %"); plt.title(f"[{tag}] E6b pair-ablation"); plt.show()

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    ax[0].bar(r, law_top); ax[0].set_xlabel("pair rank"); ax[0].set_ylabel("metric B"); ax[0].set_title(f"[{tag}] top-10 pair score B")
    tot = infl.sum()
    ax[1].plot(r, np.cumsum(law_top), "o-", label="cumulative top-K")
    ax[1].axhline(tot, color="r", ls="--", label=f"total, all pairs = {tot:.3g}")
    ax[1].set_xlabel("pair rank"); ax[1].set_ylabel("cumulative metric B")
    ax[1].set_title("cumulative vs total"); ax[1].legend(); ax[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

for tag, _ in CKPTS:
    report(tag)


## Stage 7 — do the top pairs look interpretable, and how early?
For the top-3 pairs per checkpoint: PC0 of the vision head and of the text head, each shown by its top-5 nearest images and top-5 nearest corpus sentences (per-checkpoint banks). Watch whether coherent concepts appear immediately or only later.

**Heavy:** ~8 ckpts x 3 pairs x 2 heads figures, and it loads images from disk. Edit the loop to subset if needed.

In [ ]:
# per-checkpoint text/image banks for concept read-out (banks change every checkpoint)
with open(f"utils/text_descriptions/{dataset}.txt") as f:
    _all = [l.strip() for l in f]
corpus_texts = np.array([l for c in range(len(_all) // native_per_class)
                         for l in _all[(c+1)*native_per_class - sentences_per_class:(c+1)*native_per_class]])
ds_img = ImageNet(root=path + "imagenet/", split="val", transform=visualization_preprocess)
ds_sub = dataset_subset(ds_img, samples_per_class=subset_dim, tot_samples_per_class=tot_samples_per_class, seed=seed)

def _near(vec, bank, k=5):
    return ((bank / bank.norm(dim=-1, keepdim=True)) @ (vec / vec.norm())).topk(k).indices.tolist()

def show_concepts(tag, npairs=3):
    d = CK[tag]
    Mv = d["av"].sum((1,2)) + d["mv"].sum(1); Mt = d["at"].sum((1,2)) + d["mt"].sum(1)
    A, infl, kv, kt, _, _ = pair_matrices(tag)
    top = greedy_unique(infl, kv, kt, npairs)
    bv, bt = bundles(tag, "av"), bundles(tag, "at")
    for rank, (v, t) in enumerate(top, 1):
        for lab, key, bd in [(f"V{v}", v, bv), (f"T{t}", t, bt)]:
            pc = bd[key][0][0]                                   # PC0 direction of this head
            rows = _near(pc, Mv); txts = [corpus_texts[j] for j in _near(pc, Mt)]
            fig, axes = plt.subplots(1, 5, figsize=(12, 2.6))
            for a, rr in zip(axes, rows):
                img = ds_sub[rr][0]
                a.imshow(img.permute(1, 2, 0).clamp(0, 1) if hasattr(img, "permute") else img); a.axis("off")
            fig.suptitle(f"[{tag}] pair#{rank}  head {lab}  PC0\n texts: " + " | ".join(txts), fontsize=8)
            plt.tight_layout(); plt.show()

for tag, _ in CKPTS:
    show_concepts(tag, 3)
